In [ ]:
%run MAU-Net.ipynb
%run Downsampler.ipynb

import os
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd
from scipy.linalg import hadamard
from scipy.io import savemat
import cv2
import torch
import torch.nn.functional as F

dtype = torch.cuda.FloatTensor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------
# Experimental and algorithmic parameters
# ---------------------------------------------------------
ratio = 2          # sampling ratio for single-pixel imaging (SPI)
LR = 0.01           # learning rate for MAU-Net
size = 64            # spatial resolution of SPI measurements
factor = 4           # resolution upscaling factor
file_path = "measurements.xlsx"  # path to measured SPI signals
num_iter = 51        # number of optimization iterations
show_every = 10

# ---------------------------------------------------------
# Measurement patterns
# ---------------------------------------------------------
shape = (size, size)
num = int(shape[0] * shape[1] * ratio)
seed = 123
rng = np.random.default_rng(seed)
p = rng.random((num, 1, shape[0], shape[1]))
p_pos = (p > 0.5).astype(np.float64)
p_neg = 1 - p_pos
P = p_pos - p_neg
P_torch = torch.from_numpy(P).type(dtype).to(device)

# ---------------------------------------------------------
# Load and normalize measured signals
# ---------------------------------------------------------
mear_data = pd.read_excel(file_path, header=None).values
signal = mear_data[0::2, :] - mear_data[1::2, :]
signal = signal[:num]
signal = (signal - np.mean(signal)) / np.std(signal)
signal_torch = torch.from_numpy(signal).type(dtype)
signal_torch = signal_torch.view(1, -1, 1, 1).to(device)

# ---------------------------------------------------------
# Network and operator initialization
# ---------------------------------------------------------
model = UNetWithAttention(
    input_depth=1,
    output_depth=1,
    num_channels_down=[128, 128, 128, 128, 128],
    num_channels_up=[128, 128, 128, 128, 128],
    num_channels_skip=[4, 4, 4, 4, 4]
).to(device)

downsampler = Downsampler(factor=factor,mode="avg").type(dtype)

mse = torch.nn.MSELoss()

def total_variation_loss(img):
    return (
        torch.sum(torch.abs(img[:, :, :, :-1] - img[:, :, :, 1:])) +
        torch.sum(torch.abs(img[:, :, :-1, :] - img[:, :, 1:, :]))
    )

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# ---------------------------------------------------------
# Network input initialization
# ---------------------------------------------------------
def get_input(shape):
    net_input = np.random.uniform(0, 1, shape) / 10.0
    return torch.from_numpy(net_input)

shape_netinput = (size * factor, size * factor)
netinput = get_input(shape_netinput).type(dtype)[None, None, :, :].to(device)

# ---------------------------------------------------------
# Optimization loop
# ---------------------------------------------------------
for i in range(num_iter):

    pred_img = model(netinput)

    pred_img_lr = downsampler(pred_img)

    signal_pred = F.conv2d(pred_img_lr, P_torch, stride=1, padding=0)
    signal_pred = (signal_pred - torch.mean(signal_pred)) / torch.std(signal_pred + 1e-8)

    loss_mse = mse(signal_pred, signal_torch)
    loss = loss_mse + 1e-6 * total_variation_loss(pred_img)
    loss = loss.float()

    obj = pred_img[0, 0].detach().cpu().numpy()
    obj = obj / np.max(obj)
    obj2 = pred_img_lr[0, 0].detach().cpu().numpy()
    obj2 = obj2 / np.max(obj2)

    if i % show_every == 0 and i != 0:
        print(f"iteration: {i:.2f}")
        plt.figure(figsize=(6, 2))

        plt.subplot(1, 2, 1)
        plt.imshow(obj)
        plt.title("HR")

        plt.subplot(1, 2, 2)
        plt.imshow(obj2)
        plt.title("ExpR")

        plt.show()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    torch.cuda.empty_cache()